# X6Y3 tune-up

Run cells individually, as in [TuneUpExample.ipynb](TuneUpExample.ipynb): setup, one experiment call, then its analysis. **Experiment cells acquire on Huracan when executed.** There is no experiment selector or acquisition loop.

The existing X6Y3 functions preserve the notebook's separate X90/X pulses and virtual-Z corrections. The generic LeeQ drive collection assumes X90 = X/2, so it cannot substitute for these calibrated gates unchanged.

The board work is paused. These cells have not been run on the board during this edit. Saved results are in [X6Y3_TuneUp_Inspection.ipynb](X6Y3_TuneUp_Inspection.ipynb).

## Load the environment

In [ ]:
import os, sys, json
from pathlib import Path
from datetime import datetime, timezone
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'

LEEQ_ROOT = Path('/local/data/projects/LeeQ-lbnl-qubic')
HARNESS_ROOT = Path('/local/data/projects/qubic_validation')
sys.path.insert(0, str(LEEQ_ROOT))
sys.path.append(str(HARNESS_ROOT / 'artifacts/x6y3/venv/lib/python3.11/site-packages'))

import numpy as np
import matplotlib.pyplot as plt
from leeq.experiments.experiments import ExperimentManager
from leeq.setups.huracan import create_huracan_setup
from leeq.setups.x6y3 import X6Y3Calibration
from leeq.experiments.x6y3 import acquire_plan, readout_plan, analyze_readout
from leeq.experiments.x6y3_tuneup import leeq_ramsey_plan, pingpong_plan, fit_ramsey, fit_pingpong

## Set up Huracan and select the qubit

In [ ]:
calibration = X6Y3Calibration(HARNESS_ROOT / 'X6Y3/configs_new/config.yaml')
manager = ExperimentManager()
if 'Huracan-X6Y3' in manager.get_available_setup_names():
    huracan = manager.get_setup('Huracan-X6Y3')
else:
    huracan = create_huracan_setup('/local/data/artifacts/zcu216/20260903T_aa01c78f_rdrv230_d4_candidate/bits_aa01c78f/channel_config.json')
    manager.register_setup(huracan)
manager.set_default_setup(huracan.name)

qubit = 0                     # 0 or 1; rerun this cell after changing
shots = 1024
center_offset_mhz = 0.0        # relative to the original notebook frequency
x90_scale = 1.0                # relative to the original X90 amplitude
output = HARNESS_ROOT / f'artifacts/x6y3/jupyter-experiments/Q{qubit}'

# Use the saved reference, or replace it by running the readout cell below.
reference = HARNESS_ROOT / f'artifacts/x6y3/leeq-reproduction/hardware/Q{qubit}-readout'
reference_manifest = json.loads((reference / 'manifest.json').read_text())
assert reference_manifest['status'] == 'complete' and reference_manifest['qubit'] == qubit
assert reference_manifest['calibration_sha256'] == calibration.sha256
readout_report = json.loads((reference / 'analysis.json').read_text())

## Readout reference

Optional fresh readout. Run the next cell to acquire; its analysis replaces the in-memory reference.

In [ ]:
readout = acquire_plan(
    calibration, huracan, readout_plan(qubit, shots=shots, blocks=2),
    output / f"readout-{datetime.now(timezone.utc):%Y%m%dT%H%M%S.%fZ}",
)

In [ ]:
readout_report = analyze_readout(readout)
(readout['output_dir'] / 'analysis.json').write_text(json.dumps(readout_report, indent=2))
readout_report

## Ramsey — coarse

First scan from the example: **10 MHz offset, 0–0.3 µs, 0.005 µs step**. Acquisition and analysis are separate cells.

In [ ]:
ramsey_coarse = acquire_plan(
    calibration, huracan,
    leeq_ramsey_plan(qubit, 0, center_offset_mhz=center_offset_mhz, x90_scale=x90_scale, shots=shots),
    output / f"ramsey-coarse-{datetime.now(timezone.utc):%Y%m%dT%H%M%S.%fZ}",
)

In [ ]:
coarse_fit = fit_ramsey(ramsey_coarse, readout_report)
(ramsey_coarse['output_dir'] / 'analysis.json').write_text(json.dumps(coarse_fit, indent=2))
{key: value for key, value in coarse_fit.items() if key not in ('mean', 'standard_error')}

In [ ]:
t = np.array([p['delay_us'] for p in ramsey_coarse['manifest']['points']])
p = {key: value['value'] for key, value in coarse_fit['parameters'].items()}
xx = np.linspace(t.min(), t.max(), 600)
plt.figure(figsize=(7, 4))
plt.errorbar(t, coarse_fit['mean'], yerr=coarse_fit['standard_error'], fmt='.', label='IQ projection')
plt.plot(xx, p['Amplitude'] * np.exp(-xx/p['Decay']) * np.sin(2*np.pi*p['Frequency']*xx+p['Phase']) + p['Offset'], label='Fit')
plt.xlabel('Delay (µs)'); plt.ylabel('IQ projection'); plt.legend(); plt.show()

Accept this fit before updating the in-memory center. If rejected, keep the last accepted center; the original YAML is untouched.

In [ ]:
assert coarse_fit['accepted'], 'Fit rejected: inspect the data; retain the previous center.'
center_offset_mhz = coarse_fit['new_center_offset_mhz']
print('Center correction (MHz):', center_offset_mhz)

## Ramsey — medium

**1 MHz offset, 0–3 µs, 0.05 µs step**. Uses the center accepted above.

In [ ]:
ramsey_medium = acquire_plan(
    calibration, huracan,
    leeq_ramsey_plan(qubit, 1, center_offset_mhz=center_offset_mhz, x90_scale=x90_scale, shots=shots),
    output / f"ramsey-medium-{datetime.now(timezone.utc):%Y%m%dT%H%M%S.%fZ}",
)

In [ ]:
medium_fit = fit_ramsey(ramsey_medium, readout_report)
(ramsey_medium['output_dir'] / 'analysis.json').write_text(json.dumps(medium_fit, indent=2))
{key: value for key, value in medium_fit.items() if key not in ('mean', 'standard_error')}

In [ ]:
t = np.array([p['delay_us'] for p in ramsey_medium['manifest']['points']])
p = {key: value['value'] for key, value in medium_fit['parameters'].items()}
xx = np.linspace(t.min(), t.max(), 600)
plt.figure(figsize=(7, 4))
plt.errorbar(t, medium_fit['mean'], yerr=medium_fit['standard_error'], fmt='.', label='IQ projection')
plt.plot(xx, p['Amplitude'] * np.exp(-xx/p['Decay']) * np.sin(2*np.pi*p['Frequency']*xx+p['Phase']) + p['Offset'], label='Fit')
plt.xlabel('Delay (µs)'); plt.ylabel('IQ projection'); plt.legend(); plt.show()

Accept this fit before updating the in-memory center. If rejected, keep the last accepted center; the original YAML is untouched.

In [ ]:
assert medium_fit['accepted'], 'Fit rejected: inspect the data; retain the previous center.'
center_offset_mhz = medium_fit['new_center_offset_mhz']
print('Center correction (MHz):', center_offset_mhz)

## Ramsey — fine

**0.1 MHz offset, 0–30 µs, 0.5 µs step**. Uses the center accepted above.

In [ ]:
ramsey_fine = acquire_plan(
    calibration, huracan,
    leeq_ramsey_plan(qubit, 2, center_offset_mhz=center_offset_mhz, x90_scale=x90_scale, shots=shots),
    output / f"ramsey-fine-{datetime.now(timezone.utc):%Y%m%dT%H%M%S.%fZ}",
)

In [ ]:
fine_fit = fit_ramsey(ramsey_fine, readout_report)
(ramsey_fine['output_dir'] / 'analysis.json').write_text(json.dumps(fine_fit, indent=2))
{key: value for key, value in fine_fit.items() if key not in ('mean', 'standard_error')}

In [ ]:
t = np.array([p['delay_us'] for p in ramsey_fine['manifest']['points']])
p = {key: value['value'] for key, value in fine_fit['parameters'].items()}
xx = np.linspace(t.min(), t.max(), 600)
plt.figure(figsize=(7, 4))
plt.errorbar(t, fine_fit['mean'], yerr=fine_fit['standard_error'], fmt='.', label='IQ projection')
plt.plot(xx, p['Amplitude'] * np.exp(-xx/p['Decay']) * np.sin(2*np.pi*p['Frequency']*xx+p['Phase']) + p['Offset'], label='Fit')
plt.xlabel('Delay (µs)'); plt.ylabel('IQ projection'); plt.legend(); plt.show()

Accept only a reliable fine fit. Q0’s previous fine scan was rejected; keeping the medium-scan center is valid.

In [ ]:
assert fine_fit['accepted'], 'Fit rejected: inspect the data; retain the previous center.'
center_offset_mhz = fine_fit['new_center_offset_mhz']
print('Center correction (MHz):', center_offset_mhz)

## X90 ping-pong

One scan of the independently calibrated 35 ns X90 pulse. Edit the scales/counts directly in the call. Inspect the candidate and run a fresh validation scan before changing `x90_scale`.

In [ ]:
pingpong_x90 = acquire_plan(
    calibration, huracan,
    pingpong_plan(qubit, 'X90', scales=[0.98, 1.0, 1.02], counts=[0, 4, 8, 12],
                  center_offset_mhz=center_offset_mhz, x90_scale=x90_scale, shots=shots,
                  blocks=2, seed=20260929),
    output / f"X90-pingpong-{datetime.now(timezone.utc):%Y%m%dT%H%M%S.%fZ}",
)

In [ ]:
x90_fit = fit_pingpong(pingpong_x90, readout_report)
(pingpong_x90['output_dir'] / 'analysis.json').write_text(json.dumps(x90_fit, indent=2))
{key: value for key, value in x90_fit.items() if key not in ('mean', 'curves')}

In [ ]:
plt.figure(figsize=(7, 4))
for curve in x90_fit['curves']:
    counts = np.array([p['pulse_count'] for p in curve['paired_points']])
    values = [p['difference'] for p in curve['paired_points']]
    line, = plt.plot(counts, values, '.', label=f"{curve['amplitude_scale']:.6f}×")
    xx = np.array([counts.min(), counts.max()])
    plt.plot(xx, curve['slope']*xx + curve['intercept'], color=line.get_color())
plt.axhline(0, color='gray', lw=.6)
plt.xlabel('Repeated X90 count'); plt.ylabel('Final-phase paired IQ projection'); plt.legend(); plt.show()

## X ping-pong

The 70 ns X pulse has its own amplitude. Keep the final X90 at `x90_scale`; change that value only after validating the X90 result.

In [ ]:
pingpong_x = acquire_plan(
    calibration, huracan,
    pingpong_plan(qubit, 'X', scales=[0.98, 1.0, 1.02], counts=[0, 2, 4, 6],
                  center_offset_mhz=center_offset_mhz, x90_scale=x90_scale, shots=shots,
                  blocks=2, seed=20260929),
    output / f"X-pingpong-{datetime.now(timezone.utc):%Y%m%dT%H%M%S.%fZ}",
)

In [ ]:
x_fit = fit_pingpong(pingpong_x, readout_report)
(pingpong_x['output_dir'] / 'analysis.json').write_text(json.dumps(x_fit, indent=2))
{key: value for key, value in x_fit.items() if key not in ('mean', 'curves')}

In [ ]:
plt.figure(figsize=(7, 4))
for curve in x_fit['curves']:
    counts = np.array([p['pulse_count'] for p in curve['paired_points']])
    values = [p['difference'] for p in curve['paired_points']]
    line, = plt.plot(counts, values, '.', label=f"{curve['amplitude_scale']:.6f}×")
    xx = np.array([counts.min(), counts.max()])
    plt.plot(xx, curve['slope']*xx + curve['intercept'], color=line.get_color())
plt.axhline(0, color='gray', lw=.6)
plt.xlabel('Repeated X count'); plt.ylabel('Final-phase paired IQ projection'); plt.legend(); plt.show()

## Validate a candidate

To compare a candidate against the original amplitude, rerun the corresponding ping-pong call with `scales=[1.0, candidate_scale]`, a fresh seed, and more repetitions if the response stays linear. Each call saves a new directory. Two scales give separate slopes without another amplitude interpolation.

The original calibration remains unchanged. All raw IQ, manifests and analysis files are saved under `output`. Interrupting the kernel preserves completed points; an in-flight batch may finish on the board.